# Excel: просадка ЧОД и Финреза в марте 2026

Отдельная диагностика **только по Excel-отчётам** Jan–Jun 2026.

Цель: понять, почему в **марте** просели `ЧОД` и `Фин. Рез.` относительно соседних месяцев.

## Что делает
1. Грузит Excel `01_Январь` … `06_Июнь_2026.xlsx`.
2. Сводит помесячные тоталы: ЧОД, Финрез, комиссии, АУР, амортизация, term/trx.
3. Считает просадку марта vs среднее остальных месяцев и vs Feb/Apr.
4. На зерне `inn+agr_id`: TOP agr по падению ЧОД/Финреза March−Feb и March−Apr.
5. Проверяет тождество `Финрез ≈ ЧОД − АУР − Амортизация`.

Lake/`final_df` не обязателен (опциональная сверка в конце, если есть CSV).


In [ ]:
import re
from decimal import Decimal, InvalidOperation
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 80)
pd.set_option('display.width', 220)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')

DATA_DIR = Path('/home/jovyan/documents/Equaring/Data')
OUTPUT_DIR = DATA_DIR
FOCUS_MONTH = '2026-03'
PEER_MONTHS = ['2026-01', '2026-02', '2026-04', '2026-05', '2026-06']

excel_reference_by_month = {
    '2026-01': DATA_DIR / '01_Январь_2026.xlsx',
    '2026-02': DATA_DIR / '02_Февраль_2026.xlsx',
    '2026-03': DATA_DIR / '03_Март_2026.xlsx',
    '2026-04': DATA_DIR / '04_Апрель_2026.xlsx',
    '2026-05': DATA_DIR / '05_Май_2026.xlsx',
    '2026-06': DATA_DIR / '06_Июнь_2026.xlsx',
}
excel_header_by_month = {
    '2026-01': 1,
    '2026-02': 1,
    '2026-03': 0,
    '2026-04': 0,
    '2026-05': 0,
    '2026-06': 0,
}

FINAL_DF_CSV_CANDIDATES = [
    DATA_DIR / 'final_df_period_2026_01_2026_07_mpos.csv',
    DATA_DIR / 'final_df_period_2026_01_2026_06_mpos.csv',
]

MONEY_METRICS = [
    'chod', 'fin_result', 'commission_from_ops', 'commission_monthly',
    'commission_total', 'int_component', 'aur', 'amortization', 'trx_sum',
]
COUNT_METRICS = ['unique_inn', 'retl_cnt', 'term_cnt', 'trx_cnt', 'agr_rows', 'agr_with_chod_gt0']


def normalize_inn_q1(v):
    if pd.isna(v):
        return None
    s = str(v).strip()
    s = re.sub(r'\.0$', '', s)
    s = re.sub(r'\D+', '', s)
    if not s:
        return None
    if len(s) == 9:
        s = s.zfill(10)
    elif len(s) == 11:
        s = s.zfill(12)
    return s if len(s) in (10, 12) else None


def normalize_agr_q1(v):
    if pd.isna(v):
        return None
    s = str(v).strip().replace('\xa0', '').replace(' ', '').replace(',', '.')
    if s in {'', 'nan', 'None'}:
        return None
    try:
        d = Decimal(s)
        if d == d.to_integral_value():
            return str(int(d))
    except (InvalidOperation, ValueError):
        pass
    s = re.sub(r'\.0$', '', s)
    return s if s not in {'', 'nan', 'None'} else None


def pick_col_robust(columns, candidates):
    cols = list(columns)
    lower = {str(c).strip().lower(): c for c in cols}
    for cand in candidates:
        key = str(cand).strip().lower()
        if key in lower:
            return lower[key]
    for c in cols:
        cl = str(c).strip().lower().replace('\n', ' ')
        for cand in candidates:
            if str(cand).strip().lower() in cl:
                return c
    return None


def to_num_series(s):
    return pd.to_numeric(
        s.astype(str)
         .str.replace('\xa0', '', regex=False)
         .str.replace(' ', '', regex=False)
         .str.replace(',', '.', regex=False),
        errors='coerce',
    )


print('Focus month:', FOCUS_MONTH)
print('Excel files:')
for m, p in excel_reference_by_month.items():
    print(f'  {m}: exists={p.exists()} header={excel_header_by_month.get(m, 0)} | {p.name}')


## 1) Загрузка Excel → agr-level + monthly totals


In [ ]:
COL_MAP = {
    'inn_col': ['ИНН', 'inn', 'c_inn'],
    'agr_col': ['ID договора', 'Номер договора', 'agr_id', 'abs_agr_id'],
    'retl_col': ['Кол-во торговых точек', 'Ко-во торговых точек', 'Количество торговых точек'],
    'term_col': ['Кол-во терминалов', 'Количество терминалов'],
    'trx_cnt_col': ['Количество операций', 'Количеств операций', 'trx_cnt'],
    'trx_sum_col': ['Сумма операций', 'Сумма опреаций', 'trx_sum'],
    'comm_ops_col': [
        'Комиссия эквайринга', 'Комиссия (% с операций)',
        'Комиссия \n(% с операций)', 'Комиссия % с операций',
    ],
    'comm_monthly_col': [
        'Комиссия в месяц', 'Комиссия CN (₽ в месяц)', 'Комиссия (₽ в месяц)',
        'Комиссия \n(₽ в месяц)', 'Комиссия (руб в месяц)',
    ],
    'comm_total_col': [
        'Общая комиссия', 'Комиссия общая', 'Итого комиссия',
        'Итоговая комиссия', 'Коммиссия эквайринга',
    ],
    'int_component_col': [
        'Комиссия МПС (IRF, ₽)', 'Комиссия МПС (IRF, р)',
        'Комиссия МПС (IRF, руб)', 'Комиссия МПС (IRF)',
    ],
    'chod_col': ['ЧОД'],
    'aur_col': ['АУР', 'AUR', 'Aur', 'Аур'],
    'amortization_col': [
        'Амортизация', 'Аморт', 'Амортизация терминалов',
        'amortization', 'Amortization', 'Амортизация, руб',
    ],
    'fin_result_col': [
        'Фин. Рез.', 'Фин.Рез.', 'Фин.рез.', 'Фин. рез.',
        'Фин рез', 'Финрез', 'Фин результат', 'Финансовый результат',
        'fin_result', 'Fin.Res.', 'FinRes',
    ],
}


def load_excel_month(report_month, excel_path, excel_header=0):
    ex = pd.read_excel(excel_path, header=excel_header)
    resolved = {k: pick_col_robust(ex.columns, v) for k, v in COL_MAP.items()}
    required = ['inn_col', 'agr_col', 'chod_col']
    missing = [k for k in required if resolved.get(k) is None]
    if missing:
        raise ValueError(
            f'{report_month}: missing {missing}. columns={list(ex.columns)}'
        )

    out = pd.DataFrame({
        'report_month': report_month,
        'inn_key': ex[resolved['inn_col']].map(normalize_inn_q1),
        'agr_id_key': ex[resolved['agr_col']].map(normalize_agr_q1),
    })

    def take(name, dest, default=np.nan):
        col = resolved.get(name)
        if col is None:
            out[dest] = default
        else:
            out[dest] = to_num_series(ex[col])

    take('retl_col', 'retl_cnt')
    take('term_col', 'term_cnt')
    take('trx_cnt_col', 'trx_cnt')
    take('trx_sum_col', 'trx_sum')
    take('comm_ops_col', 'commission_from_ops')
    take('comm_monthly_col', 'commission_monthly')
    take('comm_total_col', 'commission_total')
    take('int_component_col', 'int_component')
    take('chod_col', 'chod')
    take('aur_col', 'aur')
    take('amortization_col', 'amortization')
    take('fin_result_col', 'fin_result')

    # fill commission_total if absent
    if out['commission_total'].isna().all():
        out['commission_total'] = (
            out['commission_from_ops'].fillna(0) + out['commission_monthly'].fillna(0)
        )

    agr = (
        out.dropna(subset=['agr_id_key'])
        .groupby(['report_month', 'inn_key', 'agr_id_key'], as_index=False)
        .agg({
            'retl_cnt': 'max',
            'term_cnt': 'max',
            'trx_cnt': 'max',
            'trx_sum': 'sum',
            'commission_from_ops': 'sum',
            'commission_monthly': 'sum',
            'commission_total': 'sum',
            'int_component': 'sum',
            'chod': 'sum',
            'aur': 'sum',
            'amortization': 'sum',
            'fin_result': 'sum',
        })
    )
    return agr, resolved


excel_agr_parts = []
resolved_by_month = {}
for month, path in excel_reference_by_month.items():
    if not path.exists():
        print('SKIP missing:', month, path)
        continue
    header = int(excel_header_by_month.get(month, 0))
    agr_df, resolved = load_excel_month(month, path, excel_header=header)
    excel_agr_parts.append(agr_df)
    resolved_by_month[month] = resolved
    print(
        f'{month}: rows_agr={len(agr_df)} | chod={agr_df["chod"].fillna(0).sum():,.0f} '
        f'| fin={agr_df["fin_result"].fillna(0).sum():,.0f} | header={header}'
    )
    print('  cols:', {k: v for k, v in resolved.items() if v is not None})

if not excel_agr_parts:
    raise RuntimeError('No Excel months loaded')

excel_agr_df = pd.concat(excel_agr_parts, ignore_index=True)
print('excel_agr_df rows =', len(excel_agr_df))
display(excel_agr_df.head(3))


## 2) Помесячные тоталы Excel + просадка марта


In [ ]:
def month_totals(df):
    g = df.groupby('report_month', as_index=False).agg(
        unique_inn=('inn_key', 'nunique'),
        agr_rows=('agr_id_key', 'nunique'),
        agr_with_chod_gt0=('chod', lambda s: int((pd.to_numeric(s, errors='coerce').fillna(0) > 0).sum())),
        retl_cnt=('retl_cnt', 'sum'),
        term_cnt=('term_cnt', 'sum'),
        trx_cnt=('trx_cnt', 'sum'),
        trx_sum=('trx_sum', 'sum'),
        commission_from_ops=('commission_from_ops', 'sum'),
        commission_monthly=('commission_monthly', 'sum'),
        commission_total=('commission_total', 'sum'),
        int_component=('int_component', 'sum'),
        chod=('chod', 'sum'),
        aur=('aur', 'sum'),
        amortization=('amortization', 'sum'),
        fin_result=('fin_result', 'sum'),
    )
    # identity check components
    g['fin_result_recalc'] = g['chod'].fillna(0) - g['aur'].fillna(0) - g['amortization'].fillna(0)
    g['fin_identity_gap'] = g['fin_result'].fillna(0) - g['fin_result_recalc']
    g['chod_minus_comm_int'] = (
        g['chod'].fillna(0) - g['commission_total'].fillna(0) - g['int_component'].fillna(0)
    )
    return g.sort_values('report_month')


monthly_excel = month_totals(excel_agr_df)
print('=== Excel monthly totals ===')
display(monthly_excel)

# March vs peers
if FOCUS_MONTH not in set(monthly_excel['report_month']):
    raise RuntimeError(f'{FOCUS_MONTH} not in loaded months')

march = monthly_excel.loc[monthly_excel['report_month'] == FOCUS_MONTH].iloc[0]
peers = monthly_excel.loc[monthly_excel['report_month'].isin(PEER_MONTHS)].copy()
peer_mean = peers.select_dtypes(include=[np.number]).mean(numeric_only=True)

cmp_rows = []
for metric in ['chod', 'fin_result', 'commission_from_ops', 'commission_monthly',
               'commission_total', 'int_component', 'aur', 'amortization',
               'trx_sum', 'trx_cnt', 'term_cnt', 'unique_inn', 'agr_rows']:
    m_val = float(march[metric]) if pd.notna(march[metric]) else np.nan
    p_val = float(peer_mean[metric]) if metric in peer_mean.index and pd.notna(peer_mean[metric]) else np.nan
    feb = monthly_excel.loc[monthly_excel['report_month'] == '2026-02', metric]
    apr = monthly_excel.loc[monthly_excel['report_month'] == '2026-04', metric]
    feb_v = float(feb.iloc[0]) if len(feb) else np.nan
    apr_v = float(apr.iloc[0]) if len(apr) else np.nan
    cmp_rows.append({
        'metric': metric,
        'march': m_val,
        'peer_mean_excl_march': p_val,
        'delta_vs_peer_mean': m_val - p_val if pd.notna(m_val) and pd.notna(p_val) else np.nan,
        'pct_vs_peer_mean': (m_val / p_val - 1.0) if pd.notna(m_val) and p_val not in (0, np.nan) else np.nan,
        'feb': feb_v,
        'apr': apr_v,
        'delta_march_minus_feb': m_val - feb_v if pd.notna(m_val) and pd.notna(feb_v) else np.nan,
        'delta_march_minus_apr': m_val - apr_v if pd.notna(m_val) and pd.notna(apr_v) else np.nan,
    })

march_vs_peers = pd.DataFrame(cmp_rows)
print('=== March vs peer months (Excel) ===')
display(march_vs_peers)

# Contribution to fin_result dip vs peer mean
print('=== Decomposition of March fin_result vs peer mean ===')
# fin = chod - aur - amort
for part in ['chod', 'aur', 'amortization', 'fin_result']:
    row = march_vs_peers.loc[march_vs_peers['metric'] == part].iloc[0]
    sign = '-' if part in {'aur', 'amortization'} else ''
    print(
        f'{sign}{part}: march={row["march"]:,.0f} | peer_mean={row["peer_mean_excl_march"]:,.0f} '
        f'| delta={row["delta_vs_peer_mean"]:,.0f} ({row["pct_vs_peer_mean"]*100 if pd.notna(row["pct_vs_peer_mean"]) else np.nan:.1f}%)'
    )

chod_d = float(march_vs_peers.loc[march_vs_peers['metric']=='chod', 'delta_vs_peer_mean'].iloc[0])
aur_d = float(march_vs_peers.loc[march_vs_peers['metric']=='aur', 'delta_vs_peer_mean'].iloc[0])
am_d = float(march_vs_peers.loc[march_vs_peers['metric']=='amortization', 'delta_vs_peer_mean'].iloc[0])
fin_d = float(march_vs_peers.loc[march_vs_peers['metric']=='fin_result', 'delta_vs_peer_mean'].iloc[0])
# d(fin) ≈ d(chod) - d(aur) - d(amort)
print(
    f'Check: d(chod)-d(aur)-d(amort) = {chod_d - aur_d - am_d:,.0f} '
    f'vs d(fin_result) = {fin_d:,.0f}'
)

id_gap = float(march['fin_identity_gap']) if 'fin_identity_gap' in march.index else np.nan
print(f'March Excel identity gap fin - (chod-aur-amort) = {id_gap:,.2f}')


## 3) Какие договоры дали просадку (March vs Feb / March vs Apr)

TOP agr по падению `chod` и `fin_result`.  
Также: договоры, которые были в Feb/Apr, но пропали в March (или наоборот).


In [ ]:
def pivot_metric(df, metric):
    p = (
        df.pivot_table(
            index=['inn_key', 'agr_id_key'],
            columns='report_month',
            values=metric,
            aggfunc='sum',
        )
        .reset_index()
    )
    return p


def build_pair_delta(piv, left_m, right_m, metric_name):
    """right - left (e.g. march - feb): negative = drop in March."""
    out = piv[['inn_key', 'agr_id_key']].copy()
    out[left_m] = piv[left_m] if left_m in piv.columns else np.nan
    out[right_m] = piv[right_m] if right_m in piv.columns else np.nan
    out[left_m] = pd.to_numeric(out[left_m], errors='coerce').fillna(0)
    out[right_m] = pd.to_numeric(out[right_m], errors='coerce').fillna(0)
    out['delta'] = out[right_m] - out[left_m]
    out['metric'] = metric_name
    out['pair'] = f'{right_m}_minus_{left_m}'
    return out


chod_piv = pivot_metric(excel_agr_df, 'chod')
fin_piv = pivot_metric(excel_agr_df, 'fin_result')
ops_piv = pivot_metric(excel_agr_df, 'commission_from_ops')
trx_piv = pivot_metric(excel_agr_df, 'trx_sum')

pairs = []
for metric_name, piv in [('chod', chod_piv), ('fin_result', fin_piv),
                         ('commission_from_ops', ops_piv), ('trx_sum', trx_piv)]:
    if '2026-02' in piv.columns and FOCUS_MONTH in piv.columns:
        pairs.append(build_pair_delta(piv, '2026-02', FOCUS_MONTH, metric_name))
    if '2026-04' in piv.columns and FOCUS_MONTH in piv.columns:
        pairs.append(build_pair_delta(piv, '2026-04', FOCUS_MONTH, metric_name))

pair_df = pd.concat(pairs, ignore_index=True)

print('=== TOP-20 agr: CHOD drop March vs Feb ===')
top_chod_feb = (
    pair_df.loc[(pair_df['metric'] == 'chod') & (pair_df['pair'] == f'{FOCUS_MONTH}_minus_2026-02')]
    .sort_values('delta')
    .head(20)
)
display(top_chod_feb)

print('=== TOP-20 agr: CHOD drop March vs Apr ===')
top_chod_apr = (
    pair_df.loc[(pair_df['metric'] == 'chod') & (pair_df['pair'] == f'{FOCUS_MONTH}_minus_2026-04')]
    .sort_values('delta')
    .head(20)
)
display(top_chod_apr)

print('=== TOP-20 agr: fin_result drop March vs Feb ===')
top_fin_feb = (
    pair_df.loc[(pair_df['metric'] == 'fin_result') & (pair_df['pair'] == f'{FOCUS_MONTH}_minus_2026-02')]
    .sort_values('delta')
    .head(20)
)
display(top_fin_feb)

print('=== TOP-20 agr: fin_result drop March vs Apr ===')
top_fin_apr = (
    pair_df.loc[(pair_df['metric'] == 'fin_result') & (pair_df['pair'] == f'{FOCUS_MONTH}_minus_2026-04')]
    .sort_values('delta')
    .head(20)
)
display(top_fin_apr)

# Coverage / appearance
def presence(month):
    return set(
        zip(
            excel_agr_df.loc[excel_agr_df['report_month'] == month, 'inn_key'],
            excel_agr_df.loc[excel_agr_df['report_month'] == month, 'agr_id_key'],
        )
    )

keys_feb = presence('2026-02') if '2026-02' in excel_agr_df['report_month'].values else set()
keys_mar = presence(FOCUS_MONTH)
keys_apr = presence('2026-04') if '2026-04' in excel_agr_df['report_month'].values else set()

only_feb_not_mar = keys_feb - keys_mar
only_mar_not_feb = keys_mar - keys_feb
only_apr_not_mar = keys_apr - keys_mar
only_mar_not_apr = keys_mar - keys_apr

print('=== Key coverage ===')
print(f'Feb keys={len(keys_feb)} | Mar={len(keys_mar)} | Apr={len(keys_apr)}')
print(f'in Feb not Mar: {len(only_feb_not_mar)} | in Mar not Feb: {len(only_mar_not_feb)}')
print(f'in Apr not Mar: {len(only_apr_not_mar)} | in Mar not Apr: {len(only_mar_not_apr)}')

# How much CHOD drop explained by TOP-N / disappeared keys
chod_m_f = pair_df.loc[
    (pair_df['metric'] == 'chod') & (pair_df['pair'] == f'{FOCUS_MONTH}_minus_2026-02')
].copy()
total_drop = float(chod_m_f.loc[chod_m_f['delta'] < 0, 'delta'].sum())
top20_drop = float(chod_m_f.nsmallest(20, 'delta')['delta'].sum())
print(f'CHOD March-Feb: sum of negative deltas={total_drop:,.0f} | TOP20 share={top20_drop/total_drop if total_drop else np.nan:.1%}')

# Disappeared agrs contribution (were in Feb with chod, absent in Mar)
if only_feb_not_mar:
    _gone = pd.DataFrame(list(only_feb_not_mar), columns=['inn_key', 'agr_id_key'])
    feb_gone = excel_agr_df.loc[excel_agr_df['report_month'] == '2026-02'].merge(
        _gone, on=['inn_key', 'agr_id_key'], how='inner'
    )
    print(
        f'Agr in Feb not Mar: n={len(feb_gone)} | their Feb CHOD sum={feb_gone["chod"].fillna(0).sum():,.0f} '
        f'| Feb fin={feb_gone["fin_result"].fillna(0).sum():,.0f}'
    )
    display(
        feb_gone.sort_values('chod', ascending=False)
        [['inn_key', 'agr_id_key', 'chod', 'fin_result', 'commission_from_ops', 'trx_sum', 'term_cnt']]
        .head(15)
    )


## 4) Разложение просадки ЧОД на компоненты по agr (March vs Feb)

Для TOP просевших agr смотрим, что упало сильнее: `commission_from_ops`, `commission_monthly`, `int_component`, `trx_sum`.


In [ ]:
def wide_compare(months=('2026-02', FOCUS_MONTH)):
    sub = excel_agr_df.loc[excel_agr_df['report_month'].isin(months)].copy()
    metrics = [
        'chod', 'fin_result', 'commission_from_ops', 'commission_monthly',
        'commission_total', 'int_component', 'aur', 'amortization', 'trx_sum', 'trx_cnt', 'term_cnt',
    ]
    pieces = []
    for met in metrics:
        p = sub.pivot_table(
            index=['inn_key', 'agr_id_key'],
            columns='report_month',
            values=met,
            aggfunc='sum',
        )
        for m in months:
            if m not in p.columns:
                p[m] = np.nan
        p = p.reset_index()
        p[f'{met}_left'] = pd.to_numeric(p[months[0]], errors='coerce').fillna(0)
        p[f'{met}_right'] = pd.to_numeric(p[months[1]], errors='coerce').fillna(0)
        p[f'd_{met}'] = p[f'{met}_right'] - p[f'{met}_left']
        pieces.append(p[['inn_key', 'agr_id_key', f'{met}_left', f'{met}_right', f'd_{met}']])
    out = pieces[0]
    for p in pieces[1:]:
        out = out.merge(p, on=['inn_key', 'agr_id_key'], how='outer')
    return out


comp_feb_mar = wide_compare(('2026-02', FOCUS_MONTH))
comp_top = comp_feb_mar.sort_values('d_chod').head(30)
print('=== TOP-30 agr by d_chod (March−Feb) with component deltas ===')
display(comp_top[[
    'inn_key', 'agr_id_key',
    'chod_left', 'chod_right', 'd_chod',
    'd_commission_from_ops', 'd_commission_monthly', 'd_int_component',
    'd_trx_sum', 'd_trx_cnt', 'd_aur', 'd_amortization', 'd_fin_result',
]])

# Aggregate: share of total CHOD drop explained by commission_from_ops vs other
neg = comp_feb_mar.loc[comp_feb_mar['d_chod'] < 0].copy()
print('Among agr with CHOD drop March vs Feb:')
print(f'  sum d_chod = {neg["d_chod"].sum():,.0f}')
print(f'  sum d_commission_from_ops = {neg["d_commission_from_ops"].sum():,.0f}')
print(f'  sum d_commission_monthly = {neg["d_commission_monthly"].sum():,.0f}')
print(f'  sum d_int_component = {neg["d_int_component"].sum():,.0f}')
print(f'  sum d_trx_sum = {neg["d_trx_sum"].sum():,.0f}')
print(f'  sum d_fin_result = {neg["d_fin_result"].sum():,.0f}')


## 5) (Опционально) Excel March vs lake `final_df` — тот же месяц


In [ ]:
fdf = None
fdf_src = None
for pth in FINAL_DF_CSV_CANDIDATES:
    if pth.exists():
        tmp = pd.read_csv(pth, dtype=str, low_memory=False)
        tmp['report_month'] = tmp['report_month'].astype(str).str.strip().str[:7]
        fdf = tmp.loc[tmp['report_month'] == FOCUS_MONTH].copy()
        if len(fdf):
            fdf_src = str(pth)
            break

if fdf is None or not len(fdf):
    print('SKIP lake compare: final_df CSV for March not found')
else:
    print('Lake source:', fdf_src, 'rows=', len(fdf))
    fdf['inn_key'] = fdf['inn'].map(normalize_inn_q1)
    fdf['agr_id_key'] = fdf['agr_id'].map(normalize_agr_q1)
    for c in ['chod', 'fin_result', 'commission_from_ops', 'commission_monthly',
              'aur', 'amortization', 'trx_sum', 'term_cnt']:
        if c in fdf.columns:
            fdf[c] = pd.to_numeric(fdf[c], errors='coerce')
        else:
            fdf[c] = np.nan

    lake_m = fdf.dropna(subset=['agr_id_key']).groupby(['inn_key', 'agr_id_key'], as_index=False).agg({
        'chod': 'max', 'fin_result': 'max', 'commission_from_ops': 'max',
        'commission_monthly': 'max', 'aur': 'max', 'amortization': 'max',
        'trx_sum': 'max', 'term_cnt': 'max',
    })
    ex_m = excel_agr_df.loc[excel_agr_df['report_month'] == FOCUS_MONTH].copy()
    both = ex_m.merge(lake_m, on=['inn_key', 'agr_id_key'], how='outer', suffixes=('_excel', '_lake'), indicator=True)
    for c in ['chod', 'fin_result', 'commission_from_ops', 'trx_sum']:
        both[f'd_{c}'] = both[f'{c}_lake'].fillna(0) - both[f'{c}_excel'].fillna(0)

    print('March totals Excel vs lake:')
    for c in ['chod', 'fin_result', 'commission_from_ops', 'aur', 'amortization']:
        e = float(pd.to_numeric(ex_m[c], errors='coerce').fillna(0).sum())
        l = float(pd.to_numeric(lake_m[c], errors='coerce').fillna(0).sum()) if c in lake_m.columns else np.nan
        print(f'  {c}: excel={e:,.0f} | lake={l:,.0f} | delta(lake-excel)={l-e:,.0f}')

    print('TOP-15 |d_chod| Excel vs lake (March):')
    display(
        both.assign(abs_d=both['d_chod'].abs())
        .sort_values('abs_d', ascending=False)
        [['inn_key', 'agr_id_key', 'chod_excel', 'chod_lake', 'd_chod',
          'fin_result_excel', 'fin_result_lake', 'd_fin_result', '_merge']]
        .head(15)
    )


## 5b) `final_df`: ЧОД по месяцам + расхождение AUR (lake vs Excel)

Сравниваем помесячно:
- `chod` Excel vs `chod` из `final_df`
- `aur` Excel vs `aur` из `final_df` (`retl_cnt * 1926` в озере)

Нужен CSV периода (`final_df_period_…_mpos.csv`). Impala не требуется.


In [ ]:
# 5b) final_df monthly CHOD + AUR lake vs Excel
from IPython.display import display

AUR_RATE = 1926.0

fdf_period = None
fdf_period_src = None
for pth in FINAL_DF_CSV_CANDIDATES:
    if not pth.exists():
        continue
    tmp = pd.read_csv(pth, dtype=str, low_memory=False)
    if 'report_month' not in tmp.columns:
        print('SKIP no report_month in', pth)
        continue
    tmp['report_month'] = tmp['report_month'].astype(str).str.strip().str[:7]
    fdf_period = tmp
    fdf_period_src = str(pth)
    break

if fdf_period is None or not len(fdf_period):
    raise RuntimeError(
        'final_df period CSV not found. Expected one of:\n'
        + '\n'.join(str(p) for p in FINAL_DF_CSV_CANDIDATES)
    )

print('final_df source:', fdf_period_src, '| rows=', len(fdf_period))
print('months:', sorted(fdf_period['report_month'].dropna().unique().tolist()))

fdf_period = fdf_period.copy()
fdf_period['inn_key'] = fdf_period['inn'].map(normalize_inn_q1) if 'inn' in fdf_period.columns else None
fdf_period['agr_id_key'] = fdf_period['agr_id'].map(normalize_agr_q1) if 'agr_id' in fdf_period.columns else None

num_cols = [
    'chod', 'fin_result', 'aur', 'amortization', 'retl_cnt', 'term_cnt',
    'trx_cnt', 'trx_sum', 'commission_from_ops', 'commission_monthly',
    'commission_total', 'int_component',
]
for c in num_cols:
    if c in fdf_period.columns:
        fdf_period[c] = pd.to_numeric(fdf_period[c], errors='coerce')
    else:
        fdf_period[c] = np.nan

# agr-level collapse (max), then month sum — same spirit as build_lake_agg
lake_agr = (
    fdf_period.dropna(subset=['agr_id_key', 'report_month'])
    .groupby(['report_month', 'inn_key', 'agr_id_key'], as_index=False)
    .agg({
        'chod': 'max',
        'fin_result': 'max',
        'aur': 'max',
        'amortization': 'max',
        'retl_cnt': 'max',
        'term_cnt': 'max',
        'trx_cnt': 'max',
        'trx_sum': 'max',
        'commission_from_ops': 'max',
        'commission_monthly': 'max',
        'commission_total': 'max',
        'int_component': 'max',
    })
)

monthly_lake = (
    lake_agr.groupby('report_month', as_index=False)
    .agg(
        unique_inn=('inn_key', 'nunique'),
        agr_rows=('agr_id_key', 'nunique'),
        retl_cnt=('retl_cnt', 'sum'),
        term_cnt=('term_cnt', 'sum'),
        trx_cnt=('trx_cnt', 'sum'),
        trx_sum=('trx_sum', 'sum'),
        commission_from_ops=('commission_from_ops', 'sum'),
        commission_monthly=('commission_monthly', 'sum'),
        commission_total=('commission_total', 'sum'),
        int_component=('int_component', 'sum'),
        chod=('chod', 'sum'),
        aur=('aur', 'sum'),
        amortization=('amortization', 'sum'),
        fin_result=('fin_result', 'sum'),
    )
    .sort_values('report_month')
)
monthly_lake['aur_from_retl'] = monthly_lake['retl_cnt'].fillna(0) * AUR_RATE
monthly_lake['aur_vs_retl_x_rate'] = monthly_lake['aur'].fillna(0) - monthly_lake['aur_from_retl']

print('=== final_df monthly totals (lake) ===')
display(monthly_lake)

# Align with Excel monthly (already in monthly_excel from section 2)
ex = monthly_excel.copy()
for c in ['chod', 'fin_result', 'aur', 'amortization', 'retl_cnt', 'term_cnt',
          'trx_sum', 'commission_from_ops', 'commission_monthly']:
    if c not in ex.columns:
        ex[c] = np.nan

cmp = ex.merge(
    monthly_lake,
    on='report_month',
    how='outer',
    suffixes=('_excel', '_lake'),
).sort_values('report_month')

for metric in ['chod', 'aur', 'fin_result', 'retl_cnt', 'amortization', 'commission_from_ops']:
    ce, cl = f'{metric}_excel', f'{metric}_lake'
    if ce in cmp.columns and cl in cmp.columns:
        cmp[f'delta_{metric}_lake_minus_excel'] = (
            pd.to_numeric(cmp[cl], errors='coerce').fillna(0)
            - pd.to_numeric(cmp[ce], errors='coerce').fillna(0)
        )
        cmp[f'pct_{metric}_vs_excel'] = np.where(
            pd.to_numeric(cmp[ce], errors='coerce').fillna(0).abs() > 1e-9,
            cmp[f'delta_{metric}_lake_minus_excel'] / pd.to_numeric(cmp[ce], errors='coerce'),
            np.nan,
        )

print('=== CHOD by month: Excel vs final_df ===')
chod_cmp = cmp[[
    'report_month', 'chod_excel', 'chod_lake',
    'delta_chod_lake_minus_excel', 'pct_chod_vs_excel',
]].copy()
display(chod_cmp)

print('=== AUR by month: Excel vs final_df ===')
aur_cmp = cmp[[
    'report_month',
    'aur_excel', 'aur_lake',
    'retl_cnt_excel', 'retl_cnt_lake',
    'delta_aur_lake_minus_excel', 'pct_aur_vs_excel',
]].copy()
# lake identity: aur should ≈ retl*1926
if 'aur_from_retl' in monthly_lake.columns:
    aur_cmp = aur_cmp.merge(
        monthly_lake[['report_month', 'aur_from_retl', 'aur_vs_retl_x_rate']],
        on='report_month',
        how='left',
    )
aur_cmp['aur_excel_per_retl'] = np.where(
    pd.to_numeric(aur_cmp['retl_cnt_excel'], errors='coerce').fillna(0) > 0,
    pd.to_numeric(aur_cmp['aur_excel'], errors='coerce') / pd.to_numeric(aur_cmp['retl_cnt_excel'], errors='coerce'),
    np.nan,
)
aur_cmp['aur_lake_per_retl'] = np.where(
    pd.to_numeric(aur_cmp['retl_cnt_lake'], errors='coerce').fillna(0) > 0,
    pd.to_numeric(aur_cmp['aur_lake'], errors='coerce') / pd.to_numeric(aur_cmp['retl_cnt_lake'], errors='coerce'),
    np.nan,
)
display(aur_cmp)

print('=== fin_result by month: Excel vs final_df (context) ===')
display(cmp[[
    'report_month', 'fin_result_excel', 'fin_result_lake',
    'delta_fin_result_lake_minus_excel', 'pct_fin_result_vs_excel',
]])

# Is there a March CHOD dip in lake?
if FOCUS_MONTH in set(monthly_lake['report_month']):
    lk = monthly_lake.set_index('report_month')
    m_chod = float(lk.loc[FOCUS_MONTH, 'chod'])
    peer = [m for m in PEER_MONTHS if m in lk.index]
    peer_mean_chod = float(lk.loc[peer, 'chod'].mean()) if peer else np.nan
    print('=== Lake CHOD March check ===')
    print(f'March lake CHOD = {m_chod:,.2f}')
    print(f'Peer mean lake CHOD = {peer_mean_chod:,.2f}')
    if peer_mean_chod:
        print(f'March / peer_mean = {m_chod/peer_mean_chod:.3f} ({(m_chod/peer_mean_chod-1)*100:.1f}%)')
    print('Lake CHOD by month:')
    for m, v in lk['chod'].items():
        mark = ' <-- focus' if m == FOCUS_MONTH else ''
        print(f'  {m}: {float(v):,.2f}{mark}')

# AUR summary verdict
print('=== AUR verdict ===')
for _, r in aur_cmp.iterrows():
    m = r['report_month']
    d = r.get('delta_aur_lake_minus_excel')
    pct = r.get('pct_aur_vs_excel')
    per_e = r.get('aur_excel_per_retl')
    per_l = r.get('aur_lake_per_retl')
    print(
        f'{m}: delta(lake-excel)={d:,.0f} ({pct*100 if pd.notna(pct) else float("nan"):+.1f}%) | '
        f'AUR/retl excel={per_e:,.1f} lake={per_l:,.1f} (expect lake≈{AUR_RATE:.0f})'
    )

# Save
out_lake_monthly = OUTPUT_DIR / 'final_df_monthly_chod_aur_2026.csv'
out_cmp = OUTPUT_DIR / 'excel_vs_final_df_chod_aur_by_month.csv'
monthly_lake.to_csv(out_lake_monthly, index=False, encoding='utf-8-sig')
cmp.to_csv(out_cmp, index=False, encoding='utf-8-sig')
chod_cmp.to_csv(OUTPUT_DIR / 'excel_vs_final_df_chod_by_month.csv', index=False, encoding='utf-8-sig')
aur_cmp.to_csv(OUTPUT_DIR / 'excel_vs_final_df_aur_by_month.csv', index=False, encoding='utf-8-sig')
print('Saved:', out_lake_monthly)
print('Saved:', out_cmp)


## 5c) Примеры AUR в Excel за апрель, где ставка ≠ 1926

Для ручной проверки в отчёте `04_Апрель_2026.xlsx`.

На зерне `inn + agr_id`:
- `aur_per_retl = АУР / Кол-во торговых точек`
- отбираем строки, где `|aur_per_retl - 1926| > 1` (и retl > 0)

Показываем TOP по `|АУР|` и распределение ставок.


In [ ]:
# 5c) April Excel AUR examples where rate != 1926
from IPython.display import display

AUR_RATE = 1926.0
APRIL = '2026-04'
TOL = 1.0  # rub per point

if 'excel_agr_df' not in globals() or excel_agr_df is None or not len(excel_agr_df):
    raise RuntimeError('Сначала выполни секцию 1 (excel_agr_df)')

apr = excel_agr_df.loc[excel_agr_df['report_month'] == APRIL].copy()
if not len(apr):
    raise RuntimeError(f'No Excel rows for {APRIL}')

apr['aur'] = pd.to_numeric(apr['aur'], errors='coerce')
apr['retl_cnt'] = pd.to_numeric(apr['retl_cnt'], errors='coerce')
apr['aur_per_retl'] = np.where(
    apr['retl_cnt'].fillna(0) > 0,
    apr['aur'] / apr['retl_cnt'],
    np.nan,
)
apr['delta_rate_vs_1926'] = apr['aur_per_retl'] - AUR_RATE
apr['abs_delta_rate'] = apr['delta_rate_vs_1926'].abs()

has_retl = apr['retl_cnt'].fillna(0) > 0
rate_ne = has_retl & apr['aur_per_retl'].notna() & (apr['abs_delta_rate'] > TOL)
rate_eq = has_retl & apr['aur_per_retl'].notna() & (apr['abs_delta_rate'] <= TOL)
no_retl = ~has_retl

print(f'=== Excel {APRIL} AUR rate vs {AUR_RATE:.0f} ===')
print(f'agr rows: {len(apr)}')
print(f'  retl>0 & rate≈1926: {int(rate_eq.sum())}')
print(f'  retl>0 & rate≠1926: {int(rate_ne.sum())}')
print(f'  retl=0/empty: {int(no_retl.sum())}')
print(
    f'  AUR sum where rate≠1926: {apr.loc[rate_ne, "aur"].fillna(0).sum():,.2f} '
    f'({apr.loc[rate_ne, "aur"].fillna(0).sum() / max(apr["aur"].fillna(0).sum(), 1):.1%} of April Excel AUR)'
)

# rate distribution (rounded)
apr_pos = apr.loc[has_retl & apr['aur_per_retl'].notna()].copy()
apr_pos['rate_round'] = apr_pos['aur_per_retl'].round(2)
rate_dist = (
    apr_pos.groupby('rate_round', as_index=False)
    .agg(agr_cnt=('agr_id_key', 'nunique'), aur_sum=('aur', 'sum'), retl_sum=('retl_cnt', 'sum'))
    .sort_values('aur_sum', ascending=False)
)
print('=== Distinct AUR/retl rates in April Excel (by AUR sum) ===')
display(rate_dist.head(20))

examples = (
    apr.loc[rate_ne]
    .sort_values('aur', ascending=False, key=lambda s: s.abs())
    [['inn_key', 'agr_id_key', 'retl_cnt', 'aur', 'aur_per_retl', 'delta_rate_vs_1926',
      'term_cnt', 'chod', 'fin_result']]
    .head(40)
)
print(f'=== TOP-40 agr in April Excel with |AUR/retl - {AUR_RATE:.0f}| > {TOL} ===')
print('Проверь эти ИНН/договоры глазами в 04_Апрель_2026.xlsx (колонки АУР и Кол-во торговых точек).')
display(examples)

# Also show a few with rate exactly 1926 for contrast
eq_examples = (
    apr.loc[rate_eq]
    .sort_values('aur', ascending=False)
    [['inn_key', 'agr_id_key', 'retl_cnt', 'aur', 'aur_per_retl', 'term_cnt']]
    .head(10)
)
print('=== Для сравнения: 10 agr с rate≈1926 ===')
display(eq_examples)

out_ex = OUTPUT_DIR / 'excel_april_2026_aur_rate_ne_1926_examples.csv'
out_dist = OUTPUT_DIR / 'excel_april_2026_aur_rate_distribution.csv'
examples.to_csv(out_ex, index=False, encoding='utf-8-sig')
rate_dist.to_csv(out_dist, index=False, encoding='utf-8-sig')
print('Saved:', out_ex)
print('Saved:', out_dist)


## 5d) Причина просадки CHOD в марте в Озере (`final_df`)

Excel в марте нормальный; проседает **только lake**.

Разбор:
1. Помесячные компоненты lake: `commission_from_ops`, `commission_monthly`, `int_component`, `chod`.
2. March vs Feb / March vs Apr — что упало в тоталах.
3. TOP agr в lake по падению CHOD (Mar−Feb).
4. Те же agr: Excel March vs lake March (есть ли дыра только в озере).


In [ ]:
# 5d) March lake CHOD dip root-cause
from IPython.display import display

if 'lake_agr' not in globals() or lake_agr is None or not len(lake_agr):
    raise RuntimeError('Сначала выполни 5b (нужен lake_agr из final_df)')
if 'excel_agr_df' not in globals() or excel_agr_df is None or not len(excel_agr_df):
    raise RuntimeError('Сначала выполни секцию 1 (excel_agr_df)')

COMP_COLS = [
    'commission_from_ops', 'commission_monthly', 'commission_total',
    'int_component', 'chod', 'trx_sum', 'trx_cnt', 'term_cnt', 'retl_cnt',
    'aur', 'amortization', 'fin_result',
]

# --- 1) Lake monthly component view ---
lake_comp = (
    lake_agr.groupby('report_month', as_index=False)
    .agg({c: 'sum' for c in COMP_COLS if c in lake_agr.columns})
    .sort_values('report_month')
)
# identity: chod ≈ commission_total + int_component (lake formula)
lake_comp['chod_recalc'] = (
    lake_comp.get('commission_total', 0).fillna(0)
    + lake_comp.get('int_component', 0).fillna(0)
)
lake_comp['chod_identity_gap'] = lake_comp['chod'].fillna(0) - lake_comp['chod_recalc']

print('=== Lake monthly components (CHOD drivers) ===')
display(lake_comp)

focus_months = ['2026-02', FOCUS_MONTH, '2026-04']
sub = lake_comp.loc[lake_comp['report_month'].isin(focus_months)].set_index('report_month')
print('=== Lake totals Feb / Mar / Apr ===')
display(sub)

if FOCUS_MONTH in sub.index and '2026-02' in sub.index:
    print('=== March − Feb (lake totals) ===')
    d = sub.loc[FOCUS_MONTH] - sub.loc['2026-02']
    for c in COMP_COLS + ['chod_recalc']:
        if c in d.index:
            print(f'  d_{c}: {float(d[c]):,.2f}')

if FOCUS_MONTH in sub.index and '2026-04' in sub.index:
    print('=== March − Apr (lake totals) ===')
    d = sub.loc[FOCUS_MONTH] - sub.loc['2026-04']
    for c in ['chod', 'commission_from_ops', 'commission_monthly', 'int_component', 'trx_sum', 'fin_result']:
        if c in d.index:
            print(f'  d_{c}: {float(d[c]):,.2f}')

# Which component explains CHOD drop vs Feb?
if FOCUS_MONTH in sub.index and '2026-02' in sub.index:
    d_chod = float(sub.loc[FOCUS_MONTH, 'chod'] - sub.loc['2026-02', 'chod'])
    d_ops = float(sub.loc[FOCUS_MONTH, 'commission_from_ops'] - sub.loc['2026-02', 'commission_from_ops'])
    d_mon = float(sub.loc[FOCUS_MONTH, 'commission_monthly'] - sub.loc['2026-02', 'commission_monthly'])
    d_int = float(sub.loc[FOCUS_MONTH, 'int_component'] - sub.loc['2026-02', 'int_component'])
    d_tot = float(sub.loc[FOCUS_MONTH, 'commission_total'] - sub.loc['2026-02', 'commission_total'])
    print('=== Driver share of lake CHOD drop (Mar−Feb) ===')
    print(f'd_chod={d_chod:,.0f}')
    print(f'd_commission_total={d_tot:,.0f} (ops={d_ops:,.0f} + monthly={d_mon:,.0f})')
    print(f'd_int_component={d_int:,.0f}')
    print(f'd_ops+d_mon+d_int={d_ops+d_mon+d_int:,.0f} (should ≈ d_chod if formula holds)')


# --- 2) Agr-level lake: Mar vs Feb ---
def lake_pivot(metric):
    return lake_agr.pivot_table(
        index=['inn_key', 'agr_id_key'],
        columns='report_month',
        values=metric,
        aggfunc='max',
    ).reset_index()


chod_l = lake_pivot('chod')
ops_l = lake_pivot('commission_from_ops')
int_l = lake_pivot('int_component')
trx_l = lake_pivot('trx_sum')

def pair_delta(piv, left, right, name):
    out = piv[['inn_key', 'agr_id_key']].copy()
    out[left] = pd.to_numeric(piv[left], errors='coerce').fillna(0) if left in piv.columns else 0.0
    out[right] = pd.to_numeric(piv[right], errors='coerce').fillna(0) if right in piv.columns else 0.0
    out['delta'] = out[right] - out[left]
    out['metric'] = name
    return out

lake_chod_mf = pair_delta(chod_l, '2026-02', FOCUS_MONTH, 'chod')
lake_ops_mf = pair_delta(ops_l, '2026-02', FOCUS_MONTH, 'commission_from_ops')
lake_int_mf = pair_delta(int_l, '2026-02', FOCUS_MONTH, 'int_component')
lake_trx_mf = pair_delta(trx_l, '2026-02', FOCUS_MONTH, 'trx_sum')

def _rename_pair(df, left, right, dname, lname, rname):
    return df.rename(columns={left: lname, right: rname, 'delta': dname})[
        ['inn_key', 'agr_id_key', lname, rname, dname]
    ]


top_lake_drop = (
    _rename_pair(lake_chod_mf, '2026-02', FOCUS_MONTH, 'd_chod', 'chod_feb', 'chod_mar')
    .sort_values('d_chod')
    .head(30)
    .merge(_rename_pair(lake_ops_mf, '2026-02', FOCUS_MONTH, 'd_ops', 'ops_feb', 'ops_mar'),
           on=['inn_key', 'agr_id_key'], how='left')
    .merge(_rename_pair(lake_int_mf, '2026-02', FOCUS_MONTH, 'd_int', 'int_feb', 'int_mar'),
           on=['inn_key', 'agr_id_key'], how='left')
    .merge(_rename_pair(lake_trx_mf, '2026-02', FOCUS_MONTH, 'd_trx_sum', 'trx_feb', 'trx_mar'),
           on=['inn_key', 'agr_id_key'], how='left')
)

print('=== TOP-30 agr lake CHOD drop March−Feb ===')
display(top_lake_drop[[
    'inn_key', 'agr_id_key', 'chod_feb', 'chod_mar', 'd_chod',
    'ops_feb', 'ops_mar', 'd_ops',
    'int_feb', 'int_mar', 'd_int',
    'trx_feb', 'trx_mar', 'd_trx_sum',
]])

neg = lake_chod_mf.loc[lake_chod_mf['delta'] < 0]
total_neg = float(neg['delta'].sum())
top30_neg = float(lake_chod_mf.nsmallest(30, 'delta')['delta'].sum())
print(f'sum negative d_chod (all agr)={total_neg:,.0f} | TOP30 share={top30_neg/total_neg if total_neg else np.nan:.1%}')


# --- 3) For TOP lake droppers: what does Excel say in March? ---
ex_mar = excel_agr_df.loc[excel_agr_df['report_month'] == FOCUS_MONTH][
    ['inn_key', 'agr_id_key', 'chod', 'commission_from_ops', 'int_component', 'trx_sum', 'fin_result']
].copy()
ex_mar = ex_mar.rename(columns={
    'chod': 'chod_excel',
    'commission_from_ops': 'ops_excel',
    'int_component': 'int_excel',
    'trx_sum': 'trx_excel',
    'fin_result': 'fin_excel',
})
lk_mar = lake_agr.loc[lake_agr['report_month'] == FOCUS_MONTH][
    ['inn_key', 'agr_id_key', 'chod', 'commission_from_ops', 'int_component', 'trx_sum', 'fin_result']
].copy()
lk_mar = lk_mar.rename(columns={
    'chod': 'chod_lake',
    'commission_from_ops': 'ops_lake',
    'int_component': 'int_lake',
    'trx_sum': 'trx_lake',
    'fin_result': 'fin_lake',
})

top_vs_excel = (
    top_lake_drop[['inn_key', 'agr_id_key', 'chod_feb', 'chod_mar', 'd_chod']]
    .rename(columns={'chod_feb': 'chod_lake_feb', 'chod_mar': 'chod_lake_mar', 'd_chod': 'd_chod_lake_mar_feb'})
    .merge(ex_mar, on=['inn_key', 'agr_id_key'], how='left')
    .merge(lk_mar, on=['inn_key', 'agr_id_key'], how='left')
)
top_vs_excel['d_chod_lake_minus_excel_mar'] = (
    top_vs_excel['chod_lake'].fillna(0) - top_vs_excel['chod_excel'].fillna(0)
)
top_vs_excel['d_ops_lake_minus_excel_mar'] = (
    top_vs_excel['ops_lake'].fillna(0) - top_vs_excel['ops_excel'].fillna(0)
)

print('=== TOP lake CHOD droppers: Excel March vs lake March ===')
print('Если chod_excel ≈ chod_lake_feb, а chod_lake_mar просел — дыра в озёрном March.')
display(top_vs_excel[[
    'inn_key', 'agr_id_key',
    'chod_lake_feb', 'chod_lake_mar', 'd_chod_lake_mar_feb',
    'chod_excel', 'd_chod_lake_minus_excel_mar',
    'ops_excel', 'ops_lake', 'd_ops_lake_minus_excel_mar',
    'trx_excel', 'trx_lake',
]])

# Aggregate March lake vs excel among all agr
both_mar = ex_mar.merge(lk_mar, on=['inn_key', 'agr_id_key'], how='outer', indicator=True)
both_mar['d_chod'] = both_mar['chod_lake'].fillna(0) - both_mar['chod_excel'].fillna(0)
both_mar['d_ops'] = both_mar['ops_lake'].fillna(0) - both_mar['ops_excel'].fillna(0)
both_mar['d_int'] = both_mar['int_lake'].fillna(0) - both_mar['int_excel'].fillna(0)
both_mar['d_trx'] = both_mar['trx_lake'].fillna(0) - both_mar['trx_excel'].fillna(0)

print('=== March coverage Excel vs lake ===')
print(both_mar['_merge'].value_counts(dropna=False).to_string())
print(
    f'March sum d_chod (lake-excel) all agr={both_mar["d_chod"].sum():,.0f} | '
    f'sum d_ops={both_mar["d_ops"].sum():,.0f} | sum d_int={both_mar["d_int"].sum():,.0f} | '
    f'sum d_trx={both_mar["d_trx"].sum():,.0f}'
)

top_gap = both_mar.sort_values('d_chod').head(30)
print('=== TOP-30 agr by lake−excel CHOD in March (most negative) ===')
display(top_gap[[
    'inn_key', 'agr_id_key', 'chod_excel', 'chod_lake', 'd_chod',
    'ops_excel', 'ops_lake', 'd_ops', 'int_excel', 'int_lake', 'd_int',
    'trx_excel', 'trx_lake', 'd_trx', '_merge',
]])

# Verdict
print('=== VERDICT (March lake CHOD) ===')
if FOCUS_MONTH in sub.index and '2026-02' in sub.index:
    # classify primary driver
    parts = {
        'commission_from_ops': d_ops,
        'commission_monthly': d_mon,
        'int_component': d_int,
    }
    # for CHOD drop, more negative = stronger driver
    primary = min(parts.items(), key=lambda kv: kv[1])
    print(f'Primary total-level driver Mar−Feb: {primary[0]} = {primary[1]:,.0f}')
    if abs(d_ops) >= abs(d_chod) * 0.5:
        print('→ Смотри trx / commission_from_ops пайплайн за март (section trx/ops).')
    elif abs(d_int) >= abs(d_chod) * 0.5:
        print('→ Смотри int_component / IRF за март.')
    else:
        print('→ Смешанный эффект; смотри TOP agr и d_ops/d_int на них.')

out_top = OUTPUT_DIR / 'lake_march_2026_top_chod_drop_vs_feb.csv'
out_gap = OUTPUT_DIR / 'lake_vs_excel_march_2026_top_chod_gap.csv'
out_comp = OUTPUT_DIR / 'lake_monthly_chod_components_2026.csv'
top_lake_drop.to_csv(out_top, index=False, encoding='utf-8-sig')
top_gap.to_csv(out_gap, index=False, encoding='utf-8-sig')
lake_comp.to_csv(out_comp, index=False, encoding='utf-8-sig')
print('Saved:', out_top)
print('Saved:', out_gap)
print('Saved:', out_comp)


## 6) Сохранение + вердикт


In [ ]:
out_monthly = OUTPUT_DIR / 'excel_monthly_chod_finrez_2026_01_2026_06.csv'
out_march_peers = OUTPUT_DIR / 'excel_march_2026_vs_peers.csv'
out_top_chod = OUTPUT_DIR / 'excel_march_2026_top_chod_drop_vs_feb.csv'
out_comp = OUTPUT_DIR / 'excel_march_2026_component_deltas_vs_feb.csv'

monthly_excel.to_csv(out_monthly, index=False, encoding='utf-8-sig')
march_vs_peers.to_csv(out_march_peers, index=False, encoding='utf-8-sig')
top_chod_feb.to_csv(out_top_chod, index=False, encoding='utf-8-sig')
comp_top.to_csv(out_comp, index=False, encoding='utf-8-sig')

print('Saved:')
print(' ', out_monthly)
print(' ', out_march_peers)
print(' ', out_top_chod)
print(' ', out_comp)

# Auto verdict
chod_pct = float(march_vs_peers.loc[march_vs_peers['metric'] == 'chod', 'pct_vs_peer_mean'].iloc[0])
fin_pct = float(march_vs_peers.loc[march_vs_peers['metric'] == 'fin_result', 'pct_vs_peer_mean'].iloc[0])
ops_pct = float(march_vs_peers.loc[march_vs_peers['metric'] == 'commission_from_ops', 'pct_vs_peer_mean'].iloc[0])
trx_pct = float(march_vs_peers.loc[march_vs_peers['metric'] == 'trx_sum', 'pct_vs_peer_mean'].iloc[0])
aur_pct = float(march_vs_peers.loc[march_vs_peers['metric'] == 'aur', 'pct_vs_peer_mean'].iloc[0])

print('=== VERDICT (Excel-only) ===')
print(f'March CHOD vs peer mean: {chod_pct*100:.1f}%')
print(f'March fin_result vs peer mean: {fin_pct*100:.1f}%')
print(f'March commission_from_ops vs peer mean: {ops_pct*100:.1f}%')
print(f'March trx_sum vs peer mean: {trx_pct*100:.1f}%')
print(f'March AUR vs peer mean: {aur_pct*100:.1f}%')

drivers = []
if pd.notna(ops_pct) and ops_pct < -0.03:
    drivers.append('commission_from_ops')
if pd.notna(trx_pct) and trx_pct < -0.03:
    drivers.append('trx_sum')
if pd.notna(aur_pct) and aur_pct > 0.03:
    drivers.append('AUR higher (worsens finrez)')
if len(only_feb_not_mar) > 50:
    drivers.append(f'missing agrs vs Feb ({len(only_feb_not_mar)})')

if not drivers:
    drivers.append('see TOP agr deltas / identity gap')

print('Likely drivers:', ', '.join(drivers))
print(
    'Finrez follows CHOD if AUR/amort stable; if finrez drops more than CHOD — check AUR/amort uptick.'
)
